In [ ]:
import os
import gzip
import glob
import h5py
import torch
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
import matplotlib.pyplot as plt
import plotly as pt
import plotly.graph_objs as go
import plotly.express as px
import scanpy as sc
import mira

In [ ]:
import logging
import warnings
mira.utils.pretty_sderr()

In [ ]:
mira.__version__,torch.__version__

In [ ]:
torch.cuda.is_available()

In [ ]:
!pwd

In [ ]:
mira

# merge samples

In [ ]:
filenames = glob.glob("/ix/djishnu/Common_Folder/Jingyu_Data/20231110_PerturbSeq/demultiplexed_samples_240109/outs/per_sample_outs/*/count/sample_filtered_feature_bc_matrix.h5")

adatas=[]
for filename in filenames:
    adatas.append(sc.read_10x_h5(filename))
    adatas[-1].var_names_make_unique()
    adatas[-1].obs['group']=filename.split('/')[-3]

adata = adatas[0].concatenate(adatas[1:],index_unique=None)

In [ ]:
adata.var_names_make_unique()

In [ ]:
adata

In [ ]:
adata.obs

In [ ]:
adata.var

In [ ]:
adata.var.index.to_list()[-12:]

In [ ]:
adata.write('/ix/djishnu/Common_Folder/Jingyu_Data/20231110_PerturbSeq/demultiplexed_samples_240109/results/multiplexed_perturbation_scRNAseq_20231117_merged.h5ad')



In [ ]:
adata[:,('D4_BATF',
         'D4_IRF4',
         'D4_IRF8',
         'D4_NTC',
         'D4_PRDM1',
         'D4_SPI1',
         'D6_BATF',
         'D6_IRF4',
         'D6_IRF8',
         'D6_NTC',
         'D6_PRDM1',
         'D6_SPI1')].to_df()

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=20)

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')

In [ ]:
adata

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4,
             multi_panel=True
            )

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
adata = adata[adata.obs.n_genes_by_counts < 6000, :]
adata = adata[adata.obs.pct_counts_mt < 10, :]

In [ ]:
adata

# MIRA

In [ ]:
adata=sc.read_h5ad("/ix/djishnu/Common_Folder/Jingyu_Data/20231110_PerturbSeq/demultiplexed_samples_240109/results/human_B_cell_20231110_PerturbSeq_scRNA_240118.merged.h5ad")

In [ ]:
rawdata = adata.X.copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
adata.layers['counts'] = rawdata

In [ ]:
sc.pp.highly_variable_genes(adata, min_disp = 0.2)

In [ ]:
# sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
# sc.pp.highly_variable_genes(adata, min_disp = 0.2)

In [ ]:
sc.pl.highly_variable_genes(adata)

In [ ]:
adata

In [ ]:
adata.var['highly_variable'].value_counts()

In [ ]:
adata.write('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/gene_data_240729.merged.h5ad')

In [ ]:
sc.tl.pca(adata)
sc.pp.neighbors(adata, n_pcs=50)
sc.tl.umap(adata, min_dist = 0.2, negative_sample_rate=0.2)
sc.pl.umap(adata, color = 'group', frameon=False)

In [ ]:
model = mira.topics.make_model(
    adata.n_obs, adata.n_vars, # helps MIRA choose reasonable values for some hyperparameters which are not tuned.
    feature_type = 'expression',
    highly_variable_key='highly_variable',
    counts_layer='counts',
#     categorical_covariates='batch'
)

In [ ]:
model.get_learning_rate_bounds(adata)

In [ ]:
model.plot_learning_rate_bounds(figsize=(7,3))

In [ ]:
model.set_learning_rates(1e-3, 0.2)

#  Hyperparameter Optimization: Gradient based

In [ ]:
topic_contributions = mira.topics.gradient_tune(model, adata)

In [ ]:
with open('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/topic_contributions.txt', 'w') as file:
    file.write('\n'.join(str(topic) for topic in topic_contributions))

In [ ]:
NUM_TOPICS = 24

mira.pl.plot_topic_contributions(topic_contributions, NUM_TOPICS)

In [ ]:
model = model.set_params(num_topics = NUM_TOPICS).fit(adata)

In [ ]:
model.save('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/mira_model_240401_gradient.pth')

# Hyperparameter Optimization: Bayesian

In [ ]:
tuner = mira.topics.BayesianTuner(
        model = model,
        n_jobs=2,
        save_name = '/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/expression_model_tuner',
        #### IMPORTANT
        min_topics = 16, max_topics = 50, # tailor for your dataset!!!!
        #### See "Notes on min_topics, max_topics" above
        #storage = mira.topics.Redis() # if using REDIS backend for more (>5) processes
)

In [ ]:
dir(tuner)

In [ ]:
tuner.fit(adata)

In [ ]:
ax = tuner.plot_intermediate_values(palette='Spectral_r',
                                   log_hue=True, figsize=(7,3))
# ax.set(ylim = (7e2, 7.7e2))

In [ ]:
tuner.plot_pareto_front(include_pruned_trials=False, label_pareto_front=True,
                       figsize = (5,5))

In [ ]:
model = tuner.fetch_best_weights()

In [ ]:
model.save('/ix/djishnu/Common_Folder/Pease_Data/Jingyu_Data/20231110_PerturbSeq/demultiplexed_samples_240109/results/mira_model_240118.pth')

In [ ]:
#load model
model = mira.topic_model.load_model('/ix/djishnu/Common_Folder/Jingyu_Data/20231110_PerturbSeq/demultiplexed_samples_240109/results/mira_model_240118.pth')


In [ ]:
#reload adata if needed
adata=sc.read_h5ad('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/gene_data_240729.merged.h5ad')

In [ ]:
adata

In [ ]:
model

In [ ]:
model.predict(adata)

In [ ]:
model.get_umap_features(adata, box_cox=0.5)
sc.pp.neighbors(adata, use_rep = 'X_umap_features', metric = 'manhattan',n_neighbors=20)
sc.tl.umap(adata, min_dist=0.25, negative_sample_rate=5,random_state=0)

In [ ]:
sc.tl.leiden(adata,resolution=0.15,random_state=6)

In [ ]:
adata.uns['leiden_colors'] = ['dodgerblue', 'firebrick', 'green', 'black']

In [ ]:
sns.set(font_scale=1)
sns.set(rc={'figure.figsize':(5,5)})

sc.set_figure_params(scanpy=True, fontsize=14, dpi_save = 350)

sc.pl.umap(adata, color=['leiden'], s = 3, show=False, frameon=False)

#plt.savefig("/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/umap_leiden.pdf")

In [ ]:
adata.write('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/gene_data_240729.merged_topics.h5ad')

# part II start

In [ ]:
adata=sc.read_h5ad("/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/gene_data_240729.merged_topics.h5ad")


In [ ]:
sns.set(font_scale=1)
sns.set(rc={'figure.figsize':(5,5)})

sc.set_figure_params(scanpy=True, fontsize=14, dpi_save = 350)

sc.pl.umap(adata, color=['leiden'], s = 3, show=False, frameon=False)


In [ ]:
# Create a mapping dictionary for leiden annotations
leiden_mapping = {
    '0': 'ActB',
    '1': 'PB',
    '2': 'GC',
    '3': 'NA'
}

# Create the new 'leiden_annotation' column by mapping the 'leiden' column
adata.obs['cell_type_annotation'] = adata.obs['leiden'].map(leiden_mapping)

In [ ]:
adata

In [ ]:
adata.obs['replicate'] = 'Rep1'

In [ ]:
adata.obs['leiden_filtered_cells'] = ~adata.obs['leiden'].isin(["3"])
adata.obs['leiden_filtered_cells'].value_counts()

In [ ]:
adata.obs['sample'] = adata.obs['group']

# Remove specified slots from adata.obs
slots_to_remove = [
    'group'
]

# Create a new AnnData object without the specified columns
adata.obs = adata.obs.drop(columns=slots_to_remove, errors='ignore')

In [ ]:
sc.pl.umap(adata, color=['leiden', "cell_type_annotation", "sample", "replicate"], s = 3, show=False, frameon=False, cmap='inferno')

In [ ]:
adata.write('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/h5_files/tf_perturbseq_rep1_post_mira.h5ad')

In [ ]:
adata = sc.read_h5ad("/media/RAIDArray/Nick/projects/human_Bcell_GRN/20231110_PerturbSeq_Exp1/h5_files/tf_perturbseq_rep1_post_mira.h5ad")

In [ ]:
adata

In [ ]:
adata.obs['n_genes_by_counts']

In [ ]:
adata.obs['total_counts']

In [ ]:
#get cell numbers per cluster
cell_numbers = adata.obs.groupby(["leiden_annotation", "group"]).apply(len)

cell_numbers

In [ ]:
cell_numbers.to_csv('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/cell_numbers_241217.csv')

# DEG analysis

In [ ]:
adata_actB=adata[adata.obs['leiden']=='0'].copy()
adata_GC=adata[adata.obs['leiden']=='2'].copy()
adata_PB=adata[adata.obs['leiden']=='1'].copy()

In [ ]:
adata_actB.obs['group']

In [ ]:
adata_actB_batf=adata_actB[adata_actB.obs['group'].isin(['D4_BATF', 'D4_NTC'])].copy()

adata_actB_irf4=adata_actB[adata_actB.obs['group'].isin(['D4_IRF4', 'D4_NTC'])].copy()

adata_actB_irf8=adata_actB[adata_actB.obs['group'].isin(['D4_IRF8', 'D4_NTC'])].copy()

adata_actB_prdm1=adata_actB[adata_actB.obs['group'].isin(['D4_PRDM1', 'D4_NTC'])].copy()

In [ ]:
adata_GC_batf=adata_GC[adata_GC.obs['group'].isin(['D6_BATF', 'D6_NTC'])].copy()

adata_GC_irf4=adata_GC[adata_GC.obs['group'].isin(['D6_IRF4', 'D6_NTC'])].copy()

adata_GC_irf8=adata_GC[adata_GC.obs['group'].isin(['D6_IRF8', 'D6_NTC'])].copy()

adata_GC_prdm1=adata_GC[adata_GC.obs['group'].isin(['D6_PRDM1', 'D6_NTC'])].copy()

In [ ]:
adata_PB_batf=adata_PB[adata_PB.obs['group'].isin(['D4_BATF', 'D4_NTC'])].copy()

adata_PB_irf4=adata_PB[adata_PB.obs['group'].isin(['D4_IRF4', 'D4_NTC'])].copy()

adata_PB_irf8=adata_PB[adata_PB.obs['group'].isin(['D4_IRF8', 'D4_NTC'])].copy()

adata_PB_prdm1=adata_PB[adata_PB.obs['group'].isin(['D4_PRDM1', 'D4_NTC'])].copy()

In [ ]:
sc.tl.rank_genes_groups(adata_actB_batf, 'group', key_added='deg_d4_batf',
                        method='t-test_overestim_var',use_raw=False,pts=True)

sc.tl.rank_genes_groups(adata_actB_irf4, 'group', key_added='deg_d4_irf4',
                        method='t-test_overestim_var',use_raw=False,pts=True)

sc.tl.rank_genes_groups(adata_actB_irf8, 'group', key_added='deg_d4_irf8',
                        method='t-test_overestim_var',use_raw=False,pts=True)

sc.tl.rank_genes_groups(adata_actB_prdm1, 'group', key_added='deg_d4_prdm1',
                        method='t-test_overestim_var',use_raw=False,pts=True)



In [ ]:
sc.tl.rank_genes_groups(adata_GC_batf, 'group', key_added='deg_GC_batf',
                        method='t-test_overestim_var',use_raw=False,pts=True)

sc.tl.rank_genes_groups(adata_GC_irf4, 'group', key_added='deg_GC_irf4',
                        method='t-test_overestim_var',use_raw=False,pts=True)

sc.tl.rank_genes_groups(adata_GC_irf8, 'group', key_added='deg_GC_irf8',
                        method='t-test_overestim_var',use_raw=False,pts=True)

sc.tl.rank_genes_groups(adata_GC_prdm1, 'group', key_added='deg_GC_prdm1',
                        method='t-test_overestim_var',use_raw=False,pts=True)


In [ ]:
sc.tl.rank_genes_groups(adata_PB_batf, 'group', key_added='deg_PB_batf',
                        method='t-test_overestim_var',use_raw=False,pts=True)

sc.tl.rank_genes_groups(adata_PB_irf4, 'group', key_added='deg_PB_irf4',
                        method='t-test_overestim_var',use_raw=False,pts=True)

sc.tl.rank_genes_groups(adata_PB_irf8, 'group', key_added='deg_PB_irf8',
                        method='t-test_overestim_var',use_raw=False,pts=True)

sc.tl.rank_genes_groups(adata_PB_prdm1, 'group', key_added='deg_PB_prdm1',
                        method='t-test_overestim_var',use_raw=False,pts=True)


In [ ]:
df_de_gene = sc.get.rank_genes_groups_df(adata_actB_batf, group=None, key="deg_d4_batf")
df_de_gene.to_csv('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/DEG/deg_d4_batf.csv',header=True,index=False)

df_de_gene = sc.get.rank_genes_groups_df(adata_actB_irf4, group=None, key="deg_d4_irf4")
df_de_gene.to_csv('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/DEG/deg_d4_irf4.csv',header=True,index=False)

df_de_gene = sc.get.rank_genes_groups_df(adata_actB_irf8, group=None, key="deg_d4_irf8")
df_de_gene.to_csv('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/DEG/deg_d4_irf8.csv',header=True,index=False)

df_de_gene = sc.get.rank_genes_groups_df(adata_actB_prdm1, group=None, key="deg_d4_prdm1")
df_de_gene.to_csv('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/DEG/deg_d4_prdm1.csv',header=True,index=False)


In [ ]:
df_de_gene = sc.get.rank_genes_groups_df(adata_GC_batf, group=None, key="deg_GC_batf")
df_de_gene.to_csv('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/DEG/deg_GC_batf.csv',header=True,index=False)

df_de_gene = sc.get.rank_genes_groups_df(adata_GC_irf4, group=None, key="deg_GC_irf4")
df_de_gene.to_csv('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/DEG/deg_GC_irf4.csv',header=True,index=False)

df_de_gene = sc.get.rank_genes_groups_df(adata_GC_irf8, group=None, key="deg_GC_irf8")
df_de_gene.to_csv('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/DEG/deg_GC_irf8.csv',header=True,index=False)

df_de_gene = sc.get.rank_genes_groups_df(adata_GC_prdm1, group=None, key="deg_GC_prdm1")
df_de_gene.to_csv('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/DEG/deg_GC_prdm1.csv',header=True,index=False)


In [ ]:
df_de_gene = sc.get.rank_genes_groups_df(adata_PB_batf, group=None, key="deg_PB_batf")
df_de_gene.to_csv('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/DEG/deg_PB_batf.csv',header=True,index=False)

df_de_gene = sc.get.rank_genes_groups_df(adata_PB_irf4, group=None, key="deg_PB_irf4")
df_de_gene.to_csv('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/DEG/deg_PB_irf4.csv',header=True,index=False)

df_de_gene = sc.get.rank_genes_groups_df(adata_PB_irf8, group=None, key="deg_PB_irf8")
df_de_gene.to_csv('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/DEG/deg_PB_irf8.csv',header=True,index=False)

df_de_gene = sc.get.rank_genes_groups_df(adata_PB_prdm1, group=None, key="deg_PB_prdm1")
df_de_gene.to_csv('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/DEG/deg_PB_prdm1.csv',header=True,index=False)


# umaps

In [ ]:
# add day
cond = []
for x in adata.obs.group:
    x = x[0:2]
    if x == 'D4':
        cond.append('D4')
    elif x == 'D6':
        cond.append('D6')

adata.obs['day'] = cond

In [ ]:
adata.obs['day']

In [ ]:
adata.obs['group']

In [ ]:
# add knockout
cond = []
for x in adata.obs.group:
    x = x.split('_')[1]
    if x == 'BATF':
        cond.append('BATF')
    elif x == 'IRF4':
        cond.append('IRF4')
    elif x == 'IRF8':
        cond.append('IRF8')
    elif x == 'NTC':
        cond.append('Control')
    elif x == 'PRDM1':
        cond.append('PRDM1')
    elif x == 'SPI1':
        cond.append('SPI1')

adata.obs['knockout'] = cond

In [ ]:
sns.set(font_scale=1)
sns.set(rc={'figure.figsize':(5,5)})

ax = sc.pl.umap(adata, size=8, show=False)
sc.pl.umap(
    adata[adata.obs.knockout == "Control"],
    size=8,
    color="day",
    ax=ax,
    show=False,
    frameon=False
)

plt.savefig("/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/umap_control.pdf")

In [ ]:
sns.set(font_scale=1)
sns.set(rc={'figure.figsize':(5,5)})

ax = sc.pl.umap(adata, size=8, show=False)
sc.pl.umap(
    adata[adata.obs.knockout == "BATF"],
    size=8,
    color="day",
    ax=ax,
    show=False,
    frameon=False
)

plt.savefig("/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/umap_batf.pdf")

In [ ]:
sns.set(font_scale=1)
sns.set(rc={'figure.figsize':(5,5)})

ax = sc.pl.umap(adata, size=8, show=False)
sc.pl.umap(
    adata[adata.obs.knockout == "IRF4"],
    size=8,
    color="day",
    ax=ax,
    show=False,
    frameon=False
)

plt.savefig("/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/umap_irf4.pdf")

In [ ]:
sns.set(font_scale=1)
sns.set(rc={'figure.figsize':(5,5)})

ax = sc.pl.umap(adata, size=8, show=False)
sc.pl.umap(
    adata[adata.obs.knockout == "IRF8" ],
    size=8,
    color="day",
    ax=ax,
    show=False,
    frameon=False
)

plt.savefig("/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/umap_irf8.pdf")

In [ ]:
sns.set(font_scale=1)
sns.set(rc={'figure.figsize':(5,5)})

ax = sc.pl.umap(adata, size=8, show=False)
sc.pl.umap(
    adata[adata.obs.knockout == "PRDM1"],
    size=8,
    color="day",
    ax=ax,
    show=False,
    frameon=False
)

plt.savefig("/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/umap_prdm1.pdf")

In [ ]:
adata.write('/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/gene_data_240729.merged_topics.h5ad')

In [ ]:
# In this example we want to show UMAPs of different cell type markers,
# with markers of a single cell type in one row
# and with a different number of markers per cell type (row)
sc.set_figure_params(scanpy=True, fontsize=14, dpi_save = 350)
sns.set(rc={'figure.figsize':(5,5)})


# Marker genes
marker_genes= {
    'preGC': ['BATF', 'IRF8','SPIB', 'AICDA', 'FCER2', 'GCSAM'],
    'PB': ['IRF4', 'PRDM1', 'XBP1', 'JCHAIN', 'MZB1', 'CD27',]
    #'Targeted TFs': ['BATF', 'IRF4', 'PRDM1', 'SPIB'],
    #'IL12': ['IL12A', 'IL12RB1', 'IL12RB2', 'STAT4'],
    #'IFNG': ['IFNG', 'IFNGR1', 'IFNGR2', 'IFNG-AS1', 'STAT1'],
    #'Inhibitors': ['SOCS1', 'SOCS3', 'CISH']
}
# Make Axes
# Number of needed rows and columns (based on the row with the most columns)
nrow=len(marker_genes)
ncol=max([len(vs) for vs in marker_genes.values()])
fig,axs=plt.subplots(nrow,ncol,figsize=(2*ncol,2*nrow))
# Plot expression for every marker on the corresponding Axes object
for row_idx,(cell_type,markers) in enumerate(marker_genes.items()):
    col_idx=0
    for marker in markers:
        ax=axs[row_idx,col_idx]
        sc.pl.umap(adata,color=marker,ax=ax,show=False,frameon=False,s=4, legend_fontsize = 2, cmap = 'bwr')
        # Add cell type as row label - here we simply add it as ylabel of
        # the first Axes object in the row
        if col_idx==0:
            # We disabled axis drawing in UMAP to have plots without background and border
            # so we need to re-enable axis to plot the ylabel
            ax.axis('on')
            ax.tick_params(
                top='off', bottom='off', left='off', right='off',
                labelleft='on', labelbottom='off')
            ax.set_ylabel(cell_type+'\n', rotation=90, fontsize=9)
            ax.set_xlabel('')
            ax.set(frame_on=False)
        col_idx+=1
    # Remove unused column Axes in the current row
    while col_idx<ncol:
        axs[row_idx,col_idx].remove()
        col_idx+=1
# Alignment within the Figure
fig.tight_layout()


plt.savefig("/ix/djishnu/peasena/tf_perturbseq/20231110_perturbseq1/results/umap_markers.pdf")